# Reinhardt Full-stack Demo — Manim scenes

Renders the concept-diagram scenes of the 15-minute full-stack demo
(see `source/conte.md`).

- Scene 1 — Opening / problem framing (0:00–1:30)
- Scene 2 — Reinhardt intro + polylithic architecture (1:30–2:30)
- Scene 9a — DI concept diagram (12:00–12:20)
- Scene 10 — Recap, comparison, CTA (13:30–15:00)

Use `-ql` for fast iteration and `-qh` for final render. Output MP4 files
land under `media/videos/manim/`.

Theme colors follow the conte memo: background `#1a1b26` (Tokyo Night),
Reinhardt accent `#f74c00`.

In [ ]:
%load_ext manim

from manim import *

config.background_color = "#1a1b26"

BG        = "#1a1b26"
FG        = "#c0caf5"
ACCENT    = "#f74c00"   # Reinhardt brand orange
RUST      = "#ce422b"
DJANGO    = "#0c4b33"
MUTED     = "#565f89"
OK        = "#9ece6a"
WARN      = "#e0af68"

# Install on macOS if missing:
#   brew install --cask font-jetbrains-mono font-inter
MONO_FONT  = "JetBrains Mono"
TITLE_FONT = "Inter"
Text.set_default(font=MONO_FONT, color=FG)

## Scene 1 — Opening / problem framing (1:30)

In [ ]:
%%manim -qh -v WARNING Scene1Opening

import numpy as np

class Scene1Opening(Scene):
    def construct(self):
        # 0:00–0:15  "Rust is fast." + speed bars
        headline = Text("Rust is fast.", font_size=72, weight=BOLD, color=FG, font=TITLE_FONT)
        self.play(FadeIn(headline, shift=UP * 0.5), run_time=1.0)
        self.wait(0.5)
        self.play(headline.animate.to_edge(UP), run_time=0.6)

        langs  = ["Rust", "Go", "Python", "Ruby"]
        speeds = [1.00, 0.72, 0.18, 0.12]
        colors = [ACCENT, "#00ADD8", "#3572A5", "#701516"]
        BAR_LEFT = -3.0
        bars = VGroup()
        for i, (name, s, c) in enumerate(zip(langs, speeds, colors)):
            y = 1.5 - i * 1.0
            label = Text(name, font_size=28).move_to([-4.5, y, 0])
            bar = Rectangle(width=0.01, height=0.5, fill_opacity=1,
                            stroke_width=0, fill_color=c)
            bar.move_to([BAR_LEFT, y, 0], aligned_edge=LEFT)
            target = bar.copy().stretch_to_fit_width(8 * s)
            target.move_to([BAR_LEFT, y, 0], aligned_edge=LEFT)
            bars.add(label, bar, target)
            self.play(FadeIn(label), Transform(bar, target), run_time=0.5)
        self.wait(1.5)
        self.play(FadeOut(*self.mobjects))

        # 0:15–0:40  crate tile rain
        subtitle = Text("But building a web app takes...", font_size=44, color=MUTED, font=TITLE_FONT)
        self.play(Write(subtitle), run_time=1.0)
        self.wait(0.3)
        self.play(subtitle.animate.to_edge(UP))

        # Row 0 (4 tiles) and row 1 (3 tiles), each row independently centered.
        ROW_Y = [1.0, -1.2]
        positions = {
            "axum":     (-4.5, ROW_Y[0]),
            "tower":    (-1.5, ROW_Y[0]),
            "sea-orm":  ( 1.5, ROW_Y[0]),
            "utoipa":   ( 4.5, ROW_Y[0]),
            "jwt":      (-3.0, ROW_Y[1]),
            "tracing":  ( 0.0, ROW_Y[1]),
            "sqlx-cli": ( 3.0, ROW_Y[1]),
        }
        tiles = {}
        tile_group = VGroup()
        for name, (x, y) in positions.items():
            t = Text(name, font_size=26, color=FG)
            box = SurroundingRectangle(t, color=MUTED, buff=0.15,
                                       stroke_width=2,
                                       fill_color=BG, fill_opacity=1.0)
            tile = VGroup(box, t).move_to([x, y, 0]).shift(UP * 6)
            tiles[name] = tile
            tile_group.add(tile)
        self.play(
            *[t.animate.shift(DOWN * 6) for t in tile_group],
            run_time=2.0, lag_ratio=0.05,
        )

        MARGIN = 0.2

        def h_arrow(src, dst):
            s = src.get_right() + RIGHT * MARGIN
            e = dst.get_left()  + LEFT  * MARGIN
            return Arrow(s, e, color=WARN, stroke_width=3,
                         buff=0.0, tip_length=0.2)

        def elbow_down_arrow(src, dst):
            s = src.get_bottom() + DOWN * MARGIN
            e = dst.get_top()    + UP   * MARGIN
            mid_y = (s[1] + e[1]) / 2
            shaft = VMobject(stroke_color=WARN, stroke_width=3)
            shaft.set_points_as_corners([
                s,
                np.array([s[0], mid_y, 0]),
                np.array([e[0], mid_y, 0]),
                e,
            ])
            tip = ArrowTriangleFilledTip(color=WARN, length=0.2).scale(1.0)
            tip.rotate(-PI / 2 - tip.tip_angle)
            tip.move_to(e)
            return VGroup(shaft, tip)

        horizontal_deps = [("axum", "tower"), ("tower", "sea-orm"), ("sea-orm", "utoipa")]
        vertical_deps   = [("tower", "jwt"), ("sea-orm", "tracing"), ("utoipa", "sqlx-cli")]

        arrows = VGroup()
        for s, d in horizontal_deps:
            arrows.add(h_arrow(tiles[s], tiles[d]))
        for s, d in vertical_deps:
            arrows.add(elbow_down_arrow(tiles[s], tiles[d]))

        self.play(LaggedStart(*[FadeIn(a) for a in arrows], lag_ratio=0.15))
        self.wait(1.5)

        # 0:40–1:10  Django side
        self.play(FadeOut(subtitle, tile_group, arrows))
        left  = Text("Rust", font_size=40, color=RUST).move_to(LEFT * 4 + UP * 2.5)
        right = Text("Django", font_size=40, color=OK).move_to(RIGHT * 4 + UP * 2.5)
        self.play(FadeIn(left), FadeIn(right))

        rust_items   = ["axum?", "sea-orm?", "utoipa?", "jwt?", "tower?"]
        django_items = ["ORM", "Admin", "Auth", "Migration", "Serializer"]
        for i, (r, d) in enumerate(zip(rust_items, django_items)):
            rt = Text(r, font_size=26, color=MUTED).move_to(LEFT * 4 + UP * (1.5 - i * 0.7))
            dt = Text(d, font_size=26, color=OK).move_to(RIGHT * 4 + UP * (1.5 - i * 0.7))
            self.play(FadeIn(rt), FadeIn(dt), run_time=0.35)
        self.wait(1.5)

        # 1:10–1:30  punch line
        self.play(FadeOut(*self.mobjects))
        punch = VGroup(
            Text("Django: 1 command.", font_size=48, color=OK, font=TITLE_FONT),
            Text("Rust: 10+ crates.",  font_size=48, color=RUST, font=TITLE_FONT),
            Text("?", font_size=140, color=ACCENT),
        ).arrange(DOWN, buff=0.4).move_to(ORIGIN)
        self.play(Write(punch[0]))
        self.play(Write(punch[1]))
        self.play(FadeIn(punch[2], scale=0.5))
        self.wait(1.5)

## Scene 2 — Reinhardt intro + polylithic architecture (1:15)

In [ ]:
%%manim -qh -v WARNING Scene2Reinhardt

import numpy as np

class Scene2Reinhardt(Scene):
    def construct(self):
        # 1:30–1:45  logo + tagline
        logo = Text("Reinhardt", font_size=88, weight=BOLD, color=ACCENT, font=TITLE_FONT)
        tag  = Text("🦀 Django's productivity, Rust's performance",
                    font_size=30, color=FG, font=TITLE_FONT).next_to(logo, DOWN, buff=0.5)
        self.play(Write(logo), run_time=1.0)
        self.play(FadeIn(tag, shift=UP * 0.3), run_time=0.8)
        self.wait(1.0)
        self.play(logo.animate.scale(0.45).to_edge(UP),
                  FadeOut(tag))

        # 1:45–2:10  hub with nodes arranged as top row / bottom row.
        HUB_R = 0.8
        hub = Circle(radius=HUB_R, color=ACCENT, fill_opacity=1.0).set_fill(ACCENT)
        hub.set_z_index(2)
        hub_label = Text("reinhardt", font_size=22, color=BG).move_to(hub)
        hub_label.set_z_index(3)
        self.play(GrowFromCenter(hub), FadeIn(hub_label))

        crates = [
            "reinhardt-core",  "reinhardt-orm",    "reinhardt-di",      "reinhardt-auth",
            "reinhardt-admin", "reinhardt-api",    "reinhardt-graphql", "reinhardt-ws",
        ]
        TOP_Y, BOT_Y = 2.6, -2.6
        COLS = [-4.8, -1.6, 1.6, 4.8]
        layout = [
            (crates[0], COLS[0], TOP_Y),
            (crates[1], COLS[1], TOP_Y),
            (crates[2], COLS[2], TOP_Y),
            (crates[3], COLS[3], TOP_Y),
            (crates[4], COLS[0], BOT_Y),
            (crates[5], COLS[1], BOT_Y),
            (crates[6], COLS[2], BOT_Y),
            (crates[7], COLS[3], BOT_Y),
        ]
        MARGIN = 0.22
        nodes, edges = VGroup(), VGroup()
        for name, x, y in layout:
            t = Text(name, font_size=18, color=FG)
            box = SurroundingRectangle(t, color=MUTED, buff=0.12,
                                       stroke_width=1.5,
                                       fill_color=BG, fill_opacity=1.0)
            node = VGroup(box, t).move_to([x, y, 0])
            node.set_z_index(2)

            is_top = y > 0
            hub_exit = (hub.get_top() + UP * MARGIN) if is_top \
                       else (hub.get_bottom() + DOWN * MARGIN)
            node_entry = (node.get_bottom() + DOWN * MARGIN) if is_top \
                         else (node.get_top() + UP * MARGIN)
            corner = np.array([node.get_center()[0], hub_exit[1], 0])
            shaft = VMobject(stroke_color=MUTED, stroke_width=1.5)
            shaft.set_points_as_corners([hub_exit, corner, node_entry])
            shaft.set_z_index(1)

            nodes.add(node); edges.add(shaft)

        self.play(LaggedStart(*[Create(e) for e in edges], lag_ratio=0.08),
                  LaggedStart(*[FadeIn(n) for n in nodes], lag_ratio=0.08),
                  run_time=2.5)

        for n in nodes:
            self.play(n[0].animate.set_color(ACCENT), run_time=0.15)
            self.play(n[0].animate.set_color(MUTED),  run_time=0.15)
        self.wait(0.5)

        # 2:10–2:45  goal strip — polls tutorial wording.
        self.play(FadeOut(hub, hub_label, nodes, edges))
        goal = Text("Build a full-stack polls app in ~13 minutes",
                    font_size=36, color=ACCENT, font=TITLE_FONT).shift(UP * 1.2)
        steps = ["Model", "Migration", "Server fn", "Reactive UI", "Form", "Admin"]
        strip = VGroup(*[Text(s, font_size=24, color=FG) for s in steps])
        strip.arrange(RIGHT, buff=0.55).next_to(goal, DOWN, buff=0.7)
        if strip.width > 12.5:
            strip.scale(12.5 / strip.width)
        arrows_between = VGroup(*[
            Arrow(strip[i].get_right(), strip[i + 1].get_left(),
                  color=MUTED, stroke_width=2, buff=0.1,
                  tip_length=0.15,
                  max_tip_length_to_length_ratio=0.3)
            for i in range(len(strip) - 1)
        ])
        ribbon = Text("Follows the official Basics Tutorial",
                      font_size=20, color="#7aa2f7", font=TITLE_FONT)
        url = Text("reinhardt-web.dev/quickstart/tutorials/basis/",
                   font_size=18, color=MUTED, font=MONO_FONT)
        ribbon_grp = VGroup(ribbon, url).arrange(DOWN, buff=0.15)
        ribbon_grp.next_to(strip, DOWN, buff=0.7)

        self.play(FadeIn(goal))
        self.play(LaggedStart(*[FadeIn(t) for t in strip], lag_ratio=0.15),
                  LaggedStart(*[GrowArrow(a) for a in arrows_between],
                              lag_ratio=0.15))
        self.play(FadeIn(ribbon_grp, shift=UP * 0.2))
        self.wait(1.8)

## Scene 3a — Project tree (~30s)

Paints the `polls_project/` directory in stages: root files first, then the four pillars (`apps/`, `server_fn/`, `shared/`, `client/`).

In [ ]:
%%manim -qh -v WARNING Scene3aTree

import numpy as np

class Scene3aTree(Scene):
    def construct(self):
        # ~30s: tree paints itself in stages.
        # Top-left tag: tutorial part.
        tag = Text("Part 1 / 7  —  Project Setup", font_size=20, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        title = Text("polls_project/", font_size=30, color=ACCENT, weight=BOLD)
        title.to_edge(UP, buff=1.0)
        self.play(Write(title), run_time=0.8)

        # Root files first.
        root_files = ["Cargo.toml", "Makefile.toml", "build.rs", "settings/"]
        roots = VGroup(*[
            Text(f"├─ {n}", font_size=22, color=FG) for n in root_files
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.18)
        roots.next_to(title, DOWN, buff=0.5).to_edge(LEFT, buff=1.2)
        self.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in roots],
                              lag_ratio=0.2), run_time=1.6)
        self.wait(0.4)

        # The four pillars and their captions.
        pillars = [
            ("src/apps/polls/",    "models, views, admin"),
            ("src/server_fn/",     "typed RPC bridge"),
            ("src/shared/types.rs", "DTOs (server + WASM)"),
            ("src/client/",        "WASM UI (router, pages)"),
        ]
        # Move root tree to upper-left, smaller, to give room.
        root_block = VGroup(title, roots)
        self.play(root_block.animate.scale(0.85).to_corner(UL, buff=0.8)
                                        .shift(DOWN * 0.6 + RIGHT * 0.2),
                  run_time=0.6)

        pillar_grp = VGroup()
        captions = VGroup()
        # 2x2 grid of pillar cards.
        coords = [(-3.4, 0.6), (3.4, 0.6), (-3.4, -2.0), (3.4, -2.0)]
        for (path, role), (x, y) in zip(pillars, coords):
            ptxt = Text(path, font_size=22, color=ACCENT, weight=BOLD)
            box = SurroundingRectangle(ptxt, color=ACCENT, buff=0.18,
                                       stroke_width=2,
                                       fill_color=BG, fill_opacity=1.0)
            card_top = VGroup(box, ptxt)
            cap = Text(role, font_size=18, color=MUTED)
            cap.next_to(card_top, DOWN, buff=0.18)
            grp = VGroup(card_top, cap).move_to([x, y, 0])
            pillar_grp.add(grp)
            captions.add(cap)

        # Fly in from edges.
        directions = [LEFT, RIGHT, LEFT, RIGHT]
        for grp, direction in zip(pillar_grp, directions):
            shifted = grp.copy().shift(direction * 6)
            grp_orig = grp.get_center()
            grp.move_to(shifted.get_center())
            self.play(grp.animate.move_to(grp_orig), run_time=0.6)
        self.wait(2.5)

        # Pulse each pillar in sequence with caption highlight.
        for grp in pillar_grp:
            box = grp[0][0]
            self.play(box.animate.set_color(WARN), run_time=0.25)
            self.play(box.animate.set_color(ACCENT), run_time=0.25)
        self.wait(1.5)

## Scene 3b — `cargo make` task dashboard (~20s)

Eight task badges (`dev`, `makemigrations`, `migrate`, `test`, `collectstatic`, `wasm-build-dev`, `showurls`, `check`) light up in sequence.

In [ ]:
%%manim -qh -v WARNING Scene3bMakeTasks

import numpy as np

class Scene3bMakeTasks(Scene):
    def construct(self):
        # ~20s: cargo make task dashboard.
        title = Text("cargo make", font_size=40, color=ACCENT, weight=BOLD)
        title.to_edge(UP, buff=0.7)
        subtitle = Text("one-stop task runner", font_size=22, color=MUTED, font=TITLE_FONT)
        subtitle.next_to(title, DOWN, buff=0.2)
        self.play(Write(title), FadeIn(subtitle))
        self.wait(0.4)

        tasks = [
            ("dev",            ACCENT),
            ("makemigrations", OK),
            ("migrate",        OK),
            ("test",           WARN),
            ("collectstatic",  FG),
            ("wasm-build-dev", ACCENT),
            ("showurls",       FG),
            ("check",          WARN),
        ]

        # 4x2 grid of badges.
        cols = [-4.5, -1.5, 1.5, 4.5]
        rows = [0.6, -1.4]
        badges = VGroup()
        coords = []
        for i, (name, color) in enumerate(tasks):
            x = cols[i % 4]
            y = rows[i // 4]
            coords.append((x, y, color))

        # Build badges at uniform width.
        max_w = 0
        cards = []
        for (name, color), (x, y, _) in zip(tasks, coords):
            t = Text(name, font_size=20, color=FG, font=MONO_FONT)
            box = RoundedRectangle(corner_radius=0.18,
                                   width=2.5, height=0.8,
                                   stroke_color=MUTED, stroke_width=2,
                                   fill_color=BG, fill_opacity=1.0)
            box.move_to([x, y, 0])
            t.move_to(box)
            cards.append((box, t, color))
            badges.add(VGroup(box, t))

        self.play(LaggedStart(
            *[FadeIn(b, shift=UP * 0.2) for b in badges],
            lag_ratio=0.12), run_time=2.5)
        self.wait(0.3)

        # Light up each badge in sequence — accent border + faint tint.
        for box, t, color in cards:
            self.play(
                box.animate.set_stroke(color=ACCENT, width=3),
                t.animate.set_color(ACCENT),
                run_time=0.35,
            )
            self.play(
                box.animate.set_stroke(color=color, width=2),
                t.animate.set_color(FG),
                run_time=0.2,
            )

        # Final caption.
        caption = Text("migrate · test · build WASM · collect static — all in one tool",
                       font_size=20, color=MUTED)
        caption.to_edge(DOWN, buff=0.7)
        self.play(FadeIn(caption))
        self.wait(1.8)

## Scene 4a — `#[model]` macro fan-out (~30s)

`#[model]` on the left, arrows fan out to synthesized symbols on the right; compile-time check stamps land on each.

In [ ]:
%%manim -qh -v WARNING Scene4aModelMacro

import numpy as np

class Scene4aModelMacro(Scene):
    def construct(self):
        # ~30s: #[model] -> synthesized symbols, with compile-time stamps.
        tag = Text("Part 2 / 7  —  Models & Database", font_size=18, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        # Left side: macro + struct stub.
        macro = Text("#[model]", font_size=34, color=ACCENT, weight=BOLD,
                     font=MONO_FONT)
        struct_lines = [
            "pub struct Question {",
            "    id: i64,",
            "    question_text: String,",
            "    pub_date: DateTime<Utc>,",
            "}",
        ]
        struct_grp = VGroup(*[
            Text(l, font_size=18, color=FG, font=MONO_FONT)
            for l in struct_lines
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.18)

        macro.next_to(struct_grp, UP, buff=0.3, aligned_edge=LEFT)
        left_block = VGroup(macro, struct_grp)
        left_box = SurroundingRectangle(left_block, color=ACCENT, buff=0.3,
                                        stroke_width=2,
                                        fill_color=BG, fill_opacity=1.0)
        left_grp = VGroup(left_box, left_block)
        left_grp.move_to([-3.8, 0, 0])

        self.play(FadeIn(left_box), Write(macro), Write(struct_grp), run_time=1.0)
        self.play(Indicate(macro, scale_factor=1.2, color=ACCENT))
        self.wait(0.4)

        # Right side: synthesized API symbols.
        symbols = [
            "Question::objects()",
            "Question::field_question_text()",
            "Question::new()",
            "Question::pk()",
        ]
        right_grp = VGroup()
        for sym in symbols:
            t = Text(sym, font_size=18, color=FG, font=MONO_FONT)
            box = SurroundingRectangle(t, color=MUTED, buff=0.16,
                                       stroke_width=1.5,
                                       fill_color=BG, fill_opacity=1.0)
            right_grp.add(VGroup(box, t))
        right_grp.arrange(DOWN, buff=0.35, aligned_edge=LEFT)
        # Normalize widths.
        max_w = max(g[0].width for g in right_grp)
        for g in right_grp:
            g[0].stretch_to_fit_width(max_w)
            g[0].move_to(g[1])
        right_grp.move_to([3.0, 0, 0])

        # Arrows fan out from macro to each symbol.
        macro_anchor = left_box.get_right()
        arrows = VGroup()
        for g in right_grp:
            tgt = g[0].get_left() + LEFT * 0.05
            a = Arrow(macro_anchor + RIGHT * 0.05, tgt,
                      color=ACCENT, stroke_width=2.5, buff=0.0,
                      tip_length=0.18,
                      max_tip_length_to_length_ratio=0.15)
            arrows.add(a)

        self.play(LaggedStart(*[GrowArrow(a) for a in arrows], lag_ratio=0.18),
                  LaggedStart(*[FadeIn(g, shift=RIGHT * 0.2) for g in right_grp],
                              lag_ratio=0.18),
                  run_time=2.0)
        self.wait(0.4)

        # Compile-time check stamps.
        stamps = VGroup()
        for g in right_grp:
            stamp = Text("✓", font_size=28, color=OK, weight=BOLD)
            stamp.next_to(g, RIGHT, buff=0.2)
            stamps.add(stamp)
        for s in stamps:
            self.play(FadeIn(s, scale=2.0), run_time=0.25)

        cap = Text("macro-synthesized, checked at compile time",
                   font_size=22, color=ACCENT, font=TITLE_FONT)
        cap.to_edge(DOWN, buff=0.6)
        self.play(Write(cap))
        self.wait(2.0)

## Scene 4b — Question / Choice ERD (~25s)

Two tables, FK arrow, `related_name = "choices"` peel-off, and a tooltip resolving `question.choices()` to `Vec<Choice>`.

In [ ]:
%%manim -qh -v WARNING Scene4bQuestionChoiceERD

import numpy as np

class Scene4bQuestionChoiceERD(Scene):
    def construct(self):
        # ~25s: ERD with FK arrow and related_name resolution.
        tag = Text("Part 2 / 7  —  ForeignKey & related_name",
                   font_size=18, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        def table(name, fields, color):
            header = Text(name, font_size=22, color=color, weight=BOLD,
                          font=MONO_FONT)
            header_box = Rectangle(width=4.0, height=0.6,
                                   stroke_color=color, stroke_width=2,
                                   fill_color=color, fill_opacity=0.25)
            header.move_to(header_box)
            head_grp = VGroup(header_box, header)
            field_lines = VGroup(*[
                Text(f, font_size=15, color=FG, font=MONO_FONT)
                for f in fields
            ]).arrange(DOWN, aligned_edge=LEFT, buff=0.15)
            body_box = Rectangle(width=4.0,
                                 height=field_lines.height + 0.4,
                                 stroke_color=MUTED, stroke_width=1.5,
                                 fill_color=BG, fill_opacity=1.0)
            body_box.next_to(head_grp, DOWN, buff=0.0)
            field_lines.move_to(body_box).align_to(body_box, LEFT).shift(RIGHT * 0.25)
            return VGroup(head_grp, body_box, field_lines)

        question_tbl = table(
            "Question",
            ["id           : i64  (PK)",
             "question_text: String",
             "pub_date     : DateTime<Utc>"],
            ACCENT,
        )
        choice_tbl = table(
            "Choice",
            ["id          : i64 (PK)",
             "question_id : FK → Question",
             "choice_text : String",
             "votes       : i32"],
            "#7aa2f7",
        )
        question_tbl.move_to([-3.8, 0.4, 0])
        choice_tbl.move_to([3.5, -0.4, 0])

        self.play(FadeIn(question_tbl), FadeIn(choice_tbl), run_time=0.8)
        self.wait(0.3)

        # FK arrow from Choice.question -> Question.id.
        # Choice.question_id row is index 1 of its field_lines.
        choice_fk_anchor = choice_tbl[2][1].get_left()  # field_lines child 1
        question_pk_anchor = question_tbl[2][0].get_right()  # field_lines child 0
        fk_arrow = Arrow(
            choice_fk_anchor + LEFT * 0.05,
            question_pk_anchor + RIGHT * 0.05,
            color=ACCENT, stroke_width=2.5, buff=0.05,
            tip_length=0.22,
            max_tip_length_to_length_ratio=0.1,
        )
        fk_label = Text("question: ForeignKeyField<Question>",
                        font_size=15, color=ACCENT, font=MONO_FONT)
        fk_label.next_to(fk_arrow, UP, buff=0.15)
        self.play(GrowArrow(fk_arrow), FadeIn(fk_label), run_time=0.8)
        self.wait(0.5)

        # Peel related_name = "choices" off the arrow onto Question side.
        rn_label = Text('related_name = "choices"',
                        font_size=16, color=WARN, font=MONO_FONT)
        rn_box = SurroundingRectangle(rn_label, color=WARN, buff=0.1,
                                      stroke_width=1.5,
                                      fill_color=BG, fill_opacity=1.0)
        rn_grp = VGroup(rn_box, rn_label)
        rn_grp.move_to(fk_arrow.get_center())
        self.play(FadeIn(rn_grp))
        self.play(rn_grp.animate.move_to(question_tbl.get_top() + UP * 0.5),
                  run_time=0.9)
        self.wait(0.4)

        # Tooltip: question.choices() -> Vec<Choice>
        tooltip = VGroup(
            Text("question.choices()", font_size=20, color=ACCENT,
                 font=MONO_FONT),
            Text("→  Vec<Choice>", font_size=20, color=OK, font=MONO_FONT),
        ).arrange(DOWN, buff=0.15)
        tip_box = SurroundingRectangle(tooltip, color=OK, buff=0.2,
                                       stroke_width=2,
                                       fill_color=BG, fill_opacity=1.0)
        tip_grp = VGroup(tip_box, tooltip)
        tip_grp.move_to([0, -2.6, 0])
        self.play(FadeIn(tip_grp, shift=UP * 0.2), run_time=0.7)
        self.wait(2.5)

## Scene 5a — `#[server_fn]` RPC bridge (~25s)

`WASM client | server_fn | Database` triptych with shared types up top. Request flows L→R, response flows back.

In [ ]:
%%manim -qh -v WARNING Scene5aServerFnRPC

import numpy as np

class Scene5aServerFnRPC(Scene):
    def construct(self):
        # ~25s: WASM client | server_fn bridge | DB. Shared types above.
        tag = Text("Part 3 / 7  —  Views and URLs",
                   font_size=18, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        def boxed(label, color, w=2.6, h=1.0, font_size=20):
            t = Text(label, font_size=font_size, color=FG, font=MONO_FONT)
            box = RoundedRectangle(corner_radius=0.18, width=w, height=h,
                                   stroke_color=color, stroke_width=2,
                                   fill_color=BG, fill_opacity=1.0)
            t.move_to(box)
            return VGroup(box, t)

        client = boxed("WASM client", "#7aa2f7", w=2.8, h=1.1)
        bridge = boxed("#[server_fn]", ACCENT, w=2.8, h=1.1, font_size=22)
        db     = boxed("Database", OK, w=2.8, h=1.1)

        client.move_to([-4.5, -0.5, 0])
        bridge.move_to([0,    -0.5, 0])
        db.move_to(    [4.5,  -0.5, 0])

        self.play(FadeIn(client), FadeIn(bridge), FadeIn(db), run_time=0.8)
        self.wait(0.3)

        # Shared types box up top.
        shared = boxed("src/shared/types.rs", "#7aa2f7",
                       w=4.5, h=0.9, font_size=20)
        shared[0].set_stroke(color="#7aa2f7", width=2)
        shared.move_to([0, 2.0, 0])
        self.play(FadeIn(shared, shift=DOWN * 0.2), run_time=0.6)

        # Arrows from client and db up to shared.
        a_client_shared = Arrow(client.get_top() + UP * 0.05,
                                shared.get_left() + LEFT * 0.05,
                                color="#7aa2f7", stroke_width=2,
                                buff=0.0, tip_length=0.18,
                                max_tip_length_to_length_ratio=0.1)
        a_db_shared = Arrow(db.get_top() + UP * 0.05,
                            shared.get_right() + RIGHT * 0.05,
                            color="#7aa2f7", stroke_width=2,
                            buff=0.0, tip_length=0.18,
                            max_tip_length_to_length_ratio=0.1)
        self.play(GrowArrow(a_client_shared), GrowArrow(a_db_shared),
                  run_time=0.6)
        self.wait(0.3)

        # Request flow client -> bridge -> db (top arc).
        req_a = Arrow(client.get_right() + RIGHT * 0.05,
                      bridge.get_left() + LEFT * 0.05,
                      color=ACCENT, stroke_width=2.5,
                      buff=0.0, tip_length=0.2,
                      max_tip_length_to_length_ratio=0.1)
        req_b = Arrow(bridge.get_right() + RIGHT * 0.05,
                      db.get_left() + LEFT * 0.05,
                      color=ACCENT, stroke_width=2.5,
                      buff=0.0, tip_length=0.2,
                      max_tip_length_to_length_ratio=0.1)
        req_lbl = Text("request", font_size=16, color=ACCENT, font=MONO_FONT)
        req_lbl.next_to(req_a, UP, buff=0.1)
        self.play(GrowArrow(req_a), GrowArrow(req_b), FadeIn(req_lbl),
                  run_time=0.7)

        # Response flow db -> bridge -> client (offset down).
        resp_a = Arrow(db.get_left() + LEFT * 0.05 + DOWN * 0.5,
                       bridge.get_right() + RIGHT * 0.05 + DOWN * 0.5,
                       color=OK, stroke_width=2.5,
                       buff=0.0, tip_length=0.2,
                       max_tip_length_to_length_ratio=0.1)
        resp_b = Arrow(bridge.get_left() + LEFT * 0.05 + DOWN * 0.5,
                       client.get_right() + RIGHT * 0.05 + DOWN * 0.5,
                       color=OK, stroke_width=2.5,
                       buff=0.0, tip_length=0.2,
                       max_tip_length_to_length_ratio=0.1)
        resp_lbl = Text("response", font_size=16, color=OK, font=MONO_FONT)
        resp_lbl.next_to(resp_a, DOWN, buff=0.1)
        self.play(GrowArrow(resp_a), GrowArrow(resp_b), FadeIn(resp_lbl),
                  run_time=0.7)
        self.wait(0.5)

        # Highlight bridge as the bridge.
        self.play(Indicate(bridge[0], color=ACCENT, scale_factor=1.1),
                  run_time=0.6)
        self.play(bridge[0].animate.set_stroke(width=4), run_time=0.3)

        cap = Text("one type definition, two sides, zero schema drift",
                   font_size=22, color="#7aa2f7", font=TITLE_FONT)
        cap.to_edge(DOWN, buff=0.5)
        self.play(Write(cap))
        self.wait(2.0)

## Scene 5b — `use_action` reactive cycle (~30s)

Four-arc cycle (`dispatch → await RPC → signal update → re-render`), `use_action` badge inside, plus a timeline below.

In [ ]:
%%manim -qh -v WARNING Scene5bUseActionCycle

import numpy as np

class Scene5bUseActionCycle(Scene):
    def construct(self):
        # ~30s: 4-arc reactive cycle + use_action badge + timeline.
        title = Text("use_action — reactive cycle",
                     font_size=28, color=ACCENT, weight=BOLD, font=TITLE_FONT)
        title.to_edge(UP, buff=0.4)
        self.add(title)

        # Circle of 4 arcs around center [0, 0.4, 0].
        center = np.array([0, 0.5, 0])
        radius = 1.6
        labels = ["dispatch", "await RPC", "signal update", "re-render"]
        arc_colors = [ACCENT, "#7aa2f7", OK, WARN]

        # Build four arcs each spanning 90 degrees with a small gap.
        gap = 4 * DEGREES
        per = 90 * DEGREES - gap
        arcs = VGroup()
        # start at top (90deg) going clockwise.
        for i in range(4):
            start_angle = (90 - i * 90) * DEGREES - gap / 2
            arc = Arc(radius=radius, start_angle=start_angle,
                      angle=-per, color=arc_colors[i], stroke_width=5)
            arc.move_arc_center_to(center)
            arcs.add(arc)

        # Place labels at midpoints of arcs (outside).
        label_positions = []
        for i in range(4):
            mid_angle = (90 - 45 - i * 90) * DEGREES
            pos = center + (radius + 0.7) * np.array(
                [np.cos(mid_angle), np.sin(mid_angle), 0])
            label_positions.append(pos)

        text_labels = VGroup()
        for label, pos, color in zip(labels, label_positions, arc_colors):
            t = Text(label, font_size=18, color=color, font=MONO_FONT)
            t.move_to(pos)
            text_labels.add(t)

        # Tip arrows at end of each arc.
        tips = VGroup()
        for i, color in enumerate(arc_colors):
            end_angle = (90 - i * 90) * DEGREES - gap / 2 - per
            end_pt = center + radius * np.array(
                [np.cos(end_angle), np.sin(end_angle), 0])
            # Tangent direction (clockwise).
            tan = np.array([np.sin(end_angle), -np.cos(end_angle), 0])
            tip = ArrowTriangleFilledTip(color=color, length=0.2)
            tip.move_to(end_pt)
            angle_to = np.arctan2(tan[1], tan[0])
            tip.rotate(angle_to - tip.tip_angle)
            tips.add(tip)

        self.play(LaggedStart(*[Create(a) for a in arcs], lag_ratio=0.15),
                  LaggedStart(*[FadeIn(t) for t in text_labels], lag_ratio=0.15),
                  run_time=2.0)
        self.play(*[FadeIn(t) for t in tips], run_time=0.4)

        # use_action badge orbiting inside.
        badge_text = Text("use_action", font_size=18, color=ACCENT,
                          font=MONO_FONT, weight=BOLD)
        badge_box = SurroundingRectangle(badge_text, color=ACCENT, buff=0.12,
                                         stroke_width=2,
                                         fill_color=BG, fill_opacity=1.0)
        badge = VGroup(badge_box, badge_text)
        badge.move_to(center)
        self.play(FadeIn(badge, scale=1.3), run_time=0.5)
        # One mini orbit pulse.
        self.play(Indicate(badge, scale_factor=1.15, color=ACCENT),
                  run_time=0.8)

        # Timeline below.
        timeline_y = -2.6
        steps = [
            "load_questions.dispatch(())",
            "await",
            "Action<Vec<QuestionInfo>, String>::Ok(..)",
            "watch { } repaints DOM",
        ]
        timeline = VGroup()
        prev = None
        for i, s in enumerate(steps):
            t = Text(s, font_size=13, color=FG, font=MONO_FONT)
            box = SurroundingRectangle(t, color=arc_colors[i], buff=0.1,
                                       stroke_width=1.5,
                                       fill_color=BG, fill_opacity=1.0)
            grp = VGroup(box, t)
            timeline.add(grp)
        timeline.arrange(RIGHT, buff=0.25)
        if timeline.width > 12.5:
            timeline.scale(12.5 / timeline.width)
        timeline.move_to([0, timeline_y, 0])

        arrows_t = VGroup()
        for i in range(len(timeline) - 1):
            a = Arrow(timeline[i].get_right(),
                      timeline[i + 1].get_left(),
                      color=MUTED, stroke_width=2,
                      buff=0.05, tip_length=0.15,
                      max_tip_length_to_length_ratio=0.12)
            arrows_t.add(a)

        for grp, a in zip(timeline, [None] + list(arrows_t)):
            if a is not None:
                self.play(GrowArrow(a), FadeIn(grp), run_time=0.4)
            else:
                self.play(FadeIn(grp), run_time=0.4)

        cap = Text("declarative reactivity, no VDOM diffing",
                   font_size=18, color=ACCENT, font=TITLE_FONT)
        cap.to_edge(DOWN, buff=0.15)
        self.play(FadeIn(cap))
        self.wait(2.0)

## Scene 6a — `form!` macro fan-out (~25s)

`form! { … }` on the left expands to HTML form, validators, submit handler, and reactive state on the right.

In [ ]:
%%manim -qh -v WARNING Scene6aFormMacro

import numpy as np

class Scene6aFormMacro(Scene):
    def construct(self):
        # ~25s: form! { ... } -> generated artifacts.
        tag = Text("Part 4 / 7  —  Forms", font_size=18, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        # Left: declarative blueprint stylized as a code block.
        decl_lines = [
            "form! {",
            "    name: VotingForm,",
            "    server_fn: submit_vote,",
            "    fields: { ... },",
            "    watch:  { ... },",
            "}",
        ]
        decl = VGroup(*[
            Text(l, font_size=18, color=FG, font=MONO_FONT)
            for l in decl_lines
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.18)
        # Highlight `form!` keyword.
        decl[0].set_color(ACCENT)
        decl_box = SurroundingRectangle(decl, color=ACCENT, buff=0.25,
                                        stroke_width=2,
                                        fill_color=BG, fill_opacity=1.0)
        left = VGroup(decl_box, decl).move_to([-3.8, 0, 0])

        self.play(FadeIn(decl_box), Write(decl), run_time=1.2)
        self.play(Indicate(decl[0], color=ACCENT, scale_factor=1.2))
        self.wait(0.3)

        # Right: 4 generated-artifact cards.
        artifacts = [
            ("HTML <form>",       OK),
            ("field validators",  WARN),
            ("submit handler",    "#7aa2f7"),
            ("reactive state",    ACCENT),
        ]
        cards = VGroup()
        for name, color in artifacts:
            t = Text(name, font_size=18, color=FG, font=MONO_FONT)
            box = RoundedRectangle(corner_radius=0.15,
                                   width=3.4, height=0.7,
                                   stroke_color=color, stroke_width=2,
                                   fill_color=BG, fill_opacity=1.0)
            t.move_to(box)
            cards.add(VGroup(box, t))
        cards.arrange(DOWN, buff=0.35)
        cards.move_to([3.2, 0, 0])

        # Arrows from blueprint to each card.
        arrows = VGroup()
        anchor = decl_box.get_right()
        for c in cards:
            tgt = c[0].get_left()
            a = Arrow(anchor + RIGHT * 0.05, tgt + LEFT * 0.05,
                      color=MUTED, stroke_width=2,
                      buff=0.0, tip_length=0.18,
                      max_tip_length_to_length_ratio=0.1)
            arrows.add(a)

        self.play(LaggedStart(*[GrowArrow(a) for a in arrows], lag_ratio=0.18),
                  LaggedStart(*[FadeIn(c, shift=RIGHT * 0.2) for c in cards],
                              lag_ratio=0.18),
                  run_time=2.0)

        # Pulse each card in sequence.
        for c in cards:
            self.play(c[0].animate.set_stroke(width=4), run_time=0.2)
            self.play(c[0].animate.set_stroke(width=2), run_time=0.15)

        cap = Text("one declaration, everything downstream generated",
                   font_size=22, color=ACCENT, font=TITLE_FONT)
        cap.to_edge(DOWN, buff=0.5)
        self.play(Write(cap))
        self.wait(1.8)

## Scene 7a — Admin: `register::<Model>(…)` (~20s)

Two `register` calls each point at a stylized browser frame (`/admin/polls/question/`, `/admin/polls/choice/`). Tally card: `✓ List ✓ Create ✓ Edit ✓ Delete ✓ Filter — 3 lines of Rust`.

In [ ]:
%%manim -qh -v WARNING Scene7aAdminRegister

import numpy as np

class Scene7aAdminRegister(Scene):
    def construct(self):
        # ~20s: admin_site.register::<X>(...) -> /admin/polls/X/
        tag = Text("Part 7 / 7  —  Admin", font_size=18, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        def code_line(text, color=FG):
            return Text(text, font_size=18, color=color, font=MONO_FONT)

        # Left: register calls, stacked.
        line_q = code_line("admin_site.register::<Question>(ModelAdmin::default());",
                           ACCENT)
        line_c = code_line("admin_site.register::<Choice>(ModelAdmin::default());",
                           ACCENT)
        code_grp = VGroup(line_q, line_c).arrange(DOWN, aligned_edge=LEFT,
                                                   buff=0.4)
        code_box = SurroundingRectangle(code_grp, color=ACCENT, buff=0.25,
                                        stroke_width=2,
                                        fill_color=BG, fill_opacity=1.0)
        code_block = VGroup(code_box, code_grp).move_to([-3.6, 0.5, 0])

        self.play(FadeIn(code_box), Write(line_q), run_time=0.8)

        # Right: stylized browser frame.
        def browser(url, rows):
            frame = RoundedRectangle(corner_radius=0.15,
                                     width=4.4, height=2.2,
                                     stroke_color=MUTED, stroke_width=2,
                                     fill_color=BG, fill_opacity=1.0)
            # title bar.
            bar = Rectangle(width=4.4, height=0.4,
                            stroke_color=MUTED, stroke_width=1.5,
                            fill_color="#24283b", fill_opacity=1.0)
            bar.move_to(frame.get_top() + DOWN * 0.2)
            # 3 dots
            dots = VGroup(*[
                Dot(radius=0.06, color=c)
                for c in [WARN, OK, "#bb9af7"]
            ]).arrange(RIGHT, buff=0.1)
            dots.move_to(bar.get_left() + RIGHT * 0.4)
            url_t = Text(url, font_size=11, color=FG, font=MONO_FONT)
            url_t.move_to(bar.get_center() + RIGHT * 0.5)
            # body content.
            body = VGroup(*[
                Text(r, font_size=11, color=FG, font=MONO_FONT)
                for r in rows
            ]).arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            body.next_to(bar, DOWN, buff=0.25, aligned_edge=LEFT)
            body.shift(RIGHT * 0.3)
            return VGroup(frame, bar, dots, url_t, body)

        b_q = browser("/admin/polls/question/",
                      ["Questions",
                       "▢  What's up?            2026-04-20",
                       "▢  Who's your favorite?  2026-04-21",
                       "[+ Add Question]"])
        b_c = browser("/admin/polls/choice/",
                      ["Choices",
                       "▢  Not much        ↪ What's up?",
                       "▢  The sky        ↪ What's up?",
                       "[+ Add Choice]"])

        b_q.move_to([3.4, 1.5, 0])
        b_c.move_to([3.4, -1.5, 0])

        # Arrow from line_q to b_q.
        a_q = Arrow(line_q.get_right() + RIGHT * 0.1,
                    b_q.get_left() + LEFT * 0.05,
                    color=ACCENT, stroke_width=2,
                    buff=0.0, tip_length=0.18,
                    max_tip_length_to_length_ratio=0.1)
        self.play(GrowArrow(a_q), FadeIn(b_q), run_time=0.8)
        self.wait(0.3)

        self.play(Write(line_c), run_time=0.5)
        a_c = Arrow(line_c.get_right() + RIGHT * 0.1,
                    b_c.get_left() + LEFT * 0.05,
                    color=ACCENT, stroke_width=2,
                    buff=0.0, tip_length=0.18,
                    max_tip_length_to_length_ratio=0.1)
        self.play(GrowArrow(a_c), FadeIn(b_c), run_time=0.8)
        self.wait(0.5)

        # Tally card at the bottom.
        tally_text = Text("✓ List   ✓ Create   ✓ Edit   ✓ Delete   ✓ Filter",
                          font_size=22, color=OK, font=MONO_FONT)
        sub = Text("3 lines of Rust", font_size=22, color=ACCENT,
                   weight=BOLD, font=MONO_FONT)
        tally = VGroup(tally_text, sub).arrange(RIGHT, buff=0.6)
        tally_box = SurroundingRectangle(tally, color=OK, buff=0.2,
                                         stroke_width=2,
                                         fill_color=BG, fill_opacity=1.0)
        tally_grp = VGroup(tally_box, tally).to_edge(DOWN, buff=0.4)
        self.play(FadeIn(tally_grp, scale=1.05), run_time=0.8)
        self.wait(1.5)

## Scene 8a — DI provider registry (~30s)

Adapted from the previous Scene 9a. Handler signature switched to `cast_vote` with `ChoiceRepository`, `AuthUser<User>`, and `Guard<IsActiveUser>`. Compile-time clock-icon arrows; missing factory shown as a red strike-through compile error.

In [ ]:
%%manim -qh -v WARNING Scene8aDIProviders

import numpy as np

class Scene8aDIProviders(Scene):
    def construct(self):
        MARGIN = 0.35
        GAP = 1.2

        tag = Text("Bonus  —  DI + Auth", font_size=18, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        # cast_vote handler signature.
        sig_lines = [
            "#[server_fn]",
            "pub async fn cast_vote(",
            "    input: VoteInput,",
            "    #[inject] choices: ChoiceRepository,",
            "    #[inject] AuthUser(user): AuthUser<User>,",
            "    #[inject] _: Guard<IsActiveUser>,",
            ") -> Result<ChoiceInfo, ServerFnError> { ... }",
        ]
        sig = VGroup(*[
            Text(line, font_size=13, color=FG, font=MONO_FONT)
            for line in sig_lines
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.36)
        sig_box = SurroundingRectangle(sig, color=FG, buff=0.22,
                                       stroke_width=2.0,
                                       fill_color=BG, fill_opacity=1.0)
        sig_grp = VGroup(sig_box, sig)

        def boxed(label, color, font_size=13):
            t = Text(label, font_size=font_size, color=FG, font=MONO_FONT)
            box = SurroundingRectangle(t, color=color, buff=0.14,
                                       stroke_width=1.5,
                                       fill_color=BG, fill_opacity=1.0)
            return VGroup(box, t)

        slot_repo  = boxed("ChoiceRepository",      ACCENT)
        slot_auth  = boxed("AuthUser<User>",        ACCENT)
        slot_guard = boxed("Guard<IsActiveUser>",   ACCENT)
        slot_w = max(slot_repo.width, slot_auth.width, slot_guard.width)
        for s in (slot_repo, slot_auth, slot_guard):
            s[0].stretch_to_fit_width(slot_w); s[0].move_to(s[1])

        p_factory   = boxed("#[injectable_factory]\n choice_repository", MUTED, font_size=12)
        p_token     = boxed("TokenAuthConfig",                           MUTED, font_size=12)
        p_is_active = boxed("IsActiveUser",                              MUTED, font_size=12)
        p_missing   = boxed("MissingProvider",                           RUST,  font_size=12)
        prov_w = max(p_factory.width, p_token.width, p_is_active.width, p_missing.width)
        for p in (p_factory, p_token, p_is_active, p_missing):
            p[0].stretch_to_fit_width(prov_w); p[0].move_to(p[1])

        # Width measurements.
        _c = VGroup(Text("DI Container", font_size=18, weight=BOLD),
                    slot_repo.copy(), slot_auth.copy(), slot_guard.copy()).arrange(DOWN, buff=0.32)
        cont_box_w = SurroundingRectangle(_c, buff=0.22).width
        _r = VGroup(Text("Provider Registry", font_size=18, weight=BOLD),
                    p_factory.copy(), p_token.copy(), p_is_active.copy()).arrange(DOWN, buff=0.32)
        reg_box_w = SurroundingRectangle(_r, buff=0.22).width

        sig_w = sig_grp.width
        total_w = sig_w + GAP + cont_box_w + GAP + reg_box_w
        left_x = -total_w / 2

        sig_cx  = left_x + sig_w / 2
        cont_lx = left_x + sig_w + GAP
        cont_cx = cont_lx + cont_box_w / 2
        reg_lx  = cont_lx + cont_box_w + GAP
        reg_cx  = reg_lx + reg_box_w / 2

        sig_grp.move_to([sig_cx, 0, 0])

        y_repo  = sig[3].get_center()[1]
        y_auth  = sig[4].get_center()[1]
        y_guard = sig[5].get_center()[1]

        slot_repo.move_to([cont_cx,  y_repo,  0])
        slot_auth.move_to([cont_cx,  y_auth,  0])
        slot_guard.move_to([cont_cx, y_guard, 0])

        container_title = Text("DI Container", font_size=18, color=ACCENT, weight=BOLD)
        container_title.move_to([cont_cx, y_repo + 0.85, 0])
        container_box = SurroundingRectangle(
            VGroup(container_title, slot_repo, slot_auth, slot_guard),
            color=ACCENT, buff=0.2, stroke_width=2,
            fill_color=BG, fill_opacity=1.0,
        )

        p_factory.move_to([reg_cx,   y_repo,  0])
        p_token.move_to([reg_cx,     y_auth,  0])
        p_is_active.move_to([reg_cx, y_guard, 0])
        # The "missing" provider sits below for the strikethrough demo.
        p_missing.move_to([reg_cx,   y_guard - 0.95, 0])

        reg_title = Text("Provider Registry", font_size=18, color=MUTED, weight=BOLD)
        reg_title.move_to([reg_cx, y_repo + 0.85, 0])
        reg_box = SurroundingRectangle(
            VGroup(reg_title, p_factory, p_token, p_is_active, p_missing),
            color=MUTED, buff=0.2, stroke_width=2,
            fill_color=BG, fill_opacity=1.0,
        )

        # Helper for arrows.
        def h_arrow(x_start, x_end, y, color, dashed=False):
            s = np.array([x_start + MARGIN, y, 0])
            e = np.array([x_end - MARGIN, y, 0])
            return Arrow(s, e, color=color, stroke_width=3,
                         buff=0.0, tip_length=0.2,
                         max_tip_length_to_length_ratio=0.3)

        sig_right = sig_box.get_right()[0]
        cont_left = container_box.get_left()[0]
        cont_right = container_box.get_right()[0]
        reg_left = reg_box.get_left()[0]

        # Animate.
        self.play(FadeIn(sig_box), Write(sig), run_time=1.2)
        self.play(sig[3].animate.set_color(ACCENT),
                  sig[4].animate.set_color(ACCENT),
                  sig[5].animate.set_color(ACCENT), run_time=0.6)
        self.wait(0.3)

        self.play(FadeIn(container_box), FadeIn(container_title),
                  FadeIn(slot_repo), FadeIn(slot_auth), FadeIn(slot_guard),
                  run_time=0.7)

        # Arrows from sig to slots.
        a1 = h_arrow(sig_right, cont_left, y_repo,  ACCENT)
        a2 = h_arrow(sig_right, cont_left, y_auth,  ACCENT)
        a3 = h_arrow(sig_right, cont_left, y_guard, ACCENT)
        self.play(FadeIn(a1), FadeIn(a2), FadeIn(a3), run_time=0.7)

        # Compile-time clock icon caption.
        clock = Text("⏱  resolved at compile time",
                     font_size=20, color=ACCENT, font=TITLE_FONT)
        clock.to_edge(DOWN, buff=0.5)
        self.play(Write(clock))
        self.wait(0.4)

        # Provider registry on the right.
        self.play(FadeIn(reg_box), FadeIn(reg_title),
                  FadeIn(p_factory), FadeIn(p_token), FadeIn(p_is_active),
                  run_time=0.7)
        r1 = h_arrow(cont_right, reg_left, y_repo,  MUTED)
        r2 = h_arrow(cont_right, reg_left, y_auth,  MUTED)
        r3 = h_arrow(cont_right, reg_left, y_guard, MUTED)
        self.play(FadeIn(r1), FadeIn(r2), FadeIn(r3), run_time=0.7)
        self.wait(0.5)

        # Missing provider strikethrough.
        self.play(FadeIn(p_missing), run_time=0.4)
        strike = Line(p_missing.get_left(), p_missing.get_right(),
                      color=RUST, stroke_width=4)
        err = Text("missing factory  →  compile error",
                   font_size=14, color=RUST, font=MONO_FONT)
        err.next_to(p_missing, DOWN, buff=0.15)
        self.play(Create(strike), FadeIn(err), run_time=0.6)
        self.wait(2.0)

## Scene 8b — Guard chain `401 → 403 → 200` (~30s)

Three packets traverse `AuthBackend → Guard<IsActiveUser> → handler body`. No-token bounces with 401, inactive bounces with 403, valid+active reaches the handler.

In [ ]:
%%manim -qh -v WARNING Scene8bGuardChain

import numpy as np

class Scene8bGuardChain(Scene):
    def construct(self):
        # ~30s: 3 packets through AuthBackend -> Guard<IsActiveUser> -> handler.
        tag = Text("Bonus  —  guard chain", font_size=18, color=MUTED, font=TITLE_FONT)
        tag.to_corner(UL, buff=0.4)
        self.add(tag)

        # Pipeline stages.
        def stage(label, color, w=2.6, h=1.0, fs=18):
            t = Text(label, font_size=fs, color=FG, font=MONO_FONT)
            box = RoundedRectangle(corner_radius=0.18, width=w, height=h,
                                   stroke_color=color, stroke_width=2,
                                   fill_color=BG, fill_opacity=1.0)
            t.move_to(box)
            return VGroup(box, t)

        auth   = stage("AuthBackend",         "#7aa2f7", w=2.6, h=0.9, fs=18)
        guard  = stage("Guard<IsActiveUser>", ACCENT,    w=3.0, h=0.9, fs=16)
        handler = stage("handler body",       OK,        w=2.6, h=0.9, fs=18)

        auth.move_to([-4.0, 0.7, 0])
        guard.move_to([0, 0.7, 0])
        handler.move_to([4.0, 0.7, 0])

        # Pipeline arrows.
        a1 = Arrow(auth.get_right() + RIGHT * 0.05,
                   guard.get_left() + LEFT * 0.05,
                   color=MUTED, stroke_width=2.5, buff=0.0,
                   tip_length=0.2, max_tip_length_to_length_ratio=0.1)
        a2 = Arrow(guard.get_right() + RIGHT * 0.05,
                   handler.get_left() + LEFT * 0.05,
                   color=MUTED, stroke_width=2.5, buff=0.0,
                   tip_length=0.2, max_tip_length_to_length_ratio=0.1)

        self.play(FadeIn(auth), FadeIn(guard), FadeIn(handler),
                  GrowArrow(a1), GrowArrow(a2), run_time=0.8)
        self.wait(0.3)

        # Inlaid snippet (small).
        snippet_lines = [
            "#[server_fn]",
            "pub async fn cast_vote(",
            "    input: VoteInput,",
            "    #[inject] AuthUser(user): AuthUser<User>,",
            "    #[inject] _: Guard<IsActiveUser>,",
            "    #[inject] choices: ChoiceRepository,",
            ") -> Result<ChoiceInfo, ServerFnError> { ... }",
        ]
        snippet = VGroup(*[
            Text(l, font_size=11, color=FG, font=MONO_FONT)
            for l in snippet_lines
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.12)
        snippet[3].set_color(ACCENT)
        snippet[4].set_color(ACCENT)
        snippet_box = SurroundingRectangle(snippet, color=MUTED, buff=0.18,
                                           stroke_width=1.5,
                                           fill_color=BG, fill_opacity=1.0)
        snippet_grp = VGroup(snippet_box, snippet)
        snippet_grp.scale(0.85)
        snippet_grp.move_to([0, -2.2, 0])
        self.play(FadeIn(snippet_grp), run_time=0.6)

        # Helper: launch packet from start, traverse stages, optionally bounce off.
        start_x = -6.2
        y0 = 0.7

        def launch_packet(label, label_color, stop_at, code, code_color):
            packet = Dot(point=[start_x, y0, 0], radius=0.13,
                         color=label_color)
            ptag = Text(label, font_size=12, color=label_color,
                        font=MONO_FONT)
            ptag.next_to(packet, UP, buff=0.12)
            ptag.add_updater(lambda m: m.next_to(packet, UP, buff=0.12))
            self.play(FadeIn(packet), FadeIn(ptag), run_time=0.3)

            # Move to auth left edge.
            self.play(packet.animate.move_to(auth.get_center()), run_time=0.6)
            if stop_at == "auth":
                # Bounce back.
                code_t = Text(code, font_size=24, color=code_color,
                              weight=BOLD, font=MONO_FONT)
                code_t.next_to(auth, DOWN, buff=0.2)
                self.play(Indicate(auth[0], color=code_color, scale_factor=1.1),
                          FadeIn(code_t), run_time=0.5)
                self.play(packet.animate.move_to([start_x, y0, 0]),
                          run_time=0.5)
                ptag.clear_updaters()
                self.play(FadeOut(packet), FadeOut(ptag), run_time=0.3)
                return code_t
            # Continue to guard.
            self.play(packet.animate.move_to(guard.get_center()), run_time=0.6)
            if stop_at == "guard":
                code_t = Text(code, font_size=24, color=code_color,
                              weight=BOLD, font=MONO_FONT)
                code_t.next_to(guard, DOWN, buff=0.2)
                self.play(Indicate(guard[0], color=code_color, scale_factor=1.1),
                          FadeIn(code_t), run_time=0.5)
                self.play(packet.animate.move_to([start_x, y0, 0]),
                          run_time=0.6)
                ptag.clear_updaters()
                self.play(FadeOut(packet), FadeOut(ptag), run_time=0.3)
                return code_t
            # Continue to handler.
            self.play(packet.animate.move_to(handler.get_center()), run_time=0.6)
            code_t = Text(code, font_size=24, color=code_color,
                          weight=BOLD, font=MONO_FONT)
            code_t.next_to(handler, UP, buff=0.2)
            self.play(Indicate(handler[0], color=code_color, scale_factor=1.1),
                      FadeIn(code_t), run_time=0.5)
            ptag.clear_updaters()
            self.play(FadeOut(packet), run_time=0.3)
            self.remove(ptag)
            return code_t

        c1 = launch_packet("no token",          MUTED, "auth",  "401", RUST)
        c2 = launch_packet("inactive user",     WARN,  "guard", "403", WARN)
        c3 = launch_packet("active + valid",    OK,    "ok",    "200", OK)

        cap = Text("401  →  403  →  200, before your handler body runs",
                   font_size=20, color=ACCENT, font=TITLE_FONT)
        cap.to_edge(DOWN, buff=0.15)
        self.play(Write(cap))
        self.wait(2.0)

## Scene 9 — Recap, comparison, CTA (~1:15)

Renumbered from Scene 10. Checklist now includes `Forms (form!)` and `DI + Auth` rows; line counter updated to ~100.

In [ ]:
%%manim -qh -v WARNING Scene9Closing

import numpy as np

class Scene9Closing(Scene):
    def construct(self):
        # 12:15–13:00  checklist + line counter.
        items = [
            "Model  (#[model])",
            "Migrations  (Rust-diffable)",
            "Server fn  (#[server_fn] + #[inject])",
            "Reactive UI  (page! + use_action + watch)",
            "Forms  (form!)",
            "Admin  (ModelAdmin)",
            "DI + Auth  (Guard<…>)",
        ]
        checklist = VGroup(*[
            Text(f"✅ {label}", font_size=24, color=OK, font=MONO_FONT)
            for label in items
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.22).to_edge(LEFT, buff=1.0)

        counter_main = Text("≈ 100 lines", font_size=46, color=ACCENT,
                            weight=BOLD, font=TITLE_FONT)
        counter_sub = Text("of your own Rust", font_size=22, color=FG, font=TITLE_FONT)
        counter = VGroup(counter_main, counter_sub).arrange(DOWN, buff=0.2)
        counter.to_edge(RIGHT, buff=1.0).shift(UP * 0.5)

        compare = VGroup(
            Text("vs. ~600 lines",
                 font_size=20, color=MUTED, font=MONO_FONT),
            Text("hand-wired Axum + sea-orm + utoipa",
                 font_size=14, color=MUTED, font=MONO_FONT),
        ).arrange(DOWN, buff=0.1)
        compare.next_to(counter, DOWN, buff=0.8)

        self.play(LaggedStart(*[FadeIn(item, shift=RIGHT * 0.3)
                                for item in checklist], lag_ratio=0.18),
                  run_time=2.0)
        self.play(Write(counter), run_time=0.7)
        self.play(FadeIn(compare, shift=UP * 0.2), run_time=0.6)
        self.wait(2.5)

        # 13:00–13:25  CTA card.
        self.play(FadeOut(checklist, counter, compare))
        cmd1 = Text("$ cargo install reinhardt-admin-cli",
                    font_size=28, color=FG, font=MONO_FONT)
        cmd2 = Text("$ reinhardt-admin startproject my_app --template pages",
                    font_size=22, color=MUTED, font=MONO_FONT)
        cmds = VGroup(cmd1, cmd2).arrange(DOWN, aligned_edge=LEFT, buff=0.3)

        site = Text("reinhardt-web.dev/quickstart/tutorials/basis/",
                    font_size=22, color=ACCENT, font=MONO_FONT)
        repo = Text("github.com/kent8192/reinhardt-web",
                    font_size=20, color=MUTED, font=MONO_FONT)
        links = VGroup(site, repo).arrange(DOWN, buff=0.2)

        block = VGroup(cmds, links).arrange(DOWN, buff=0.7)
        self.play(Write(cmds), run_time=1.0)
        self.play(FadeIn(links), run_time=0.6)
        self.wait(2.0)

        # 13:25–13:30  closing tagline.
        self.play(FadeOut(block))
        closer = Text("Django's productivity. Rust's performance.",
                      font_size=40, weight=BOLD, color=ACCENT, font=TITLE_FONT)
        self.play(FadeIn(closer, scale=0.9))
        self.wait(1.5)
        self.play(FadeOut(closer))